# Gold — Camada de consumo

Constrói as tabelas finais para análise e dashboard a partir de `silver.custo_soja`:

- **`gold.fato_custo`** — só os itens de custo classificados (remove totais e
  `Nao classificado`). O custo total passa a ser a **soma dos grupos**, calculada
  na análise, não uma linha da CONAB.
- **`gold.dim_estado`** — dimensão geográfica dos 27 estados, com a geometria
  (polígono) em GeoJSON texto, para mapas coropléticos.

**Pré-requisitos:** `silver.custo_soja` já existe (notebook 03) e o arquivo
`dados/geo/br_estados.geojson` está no projeto.

**Idempotente:** recria as tabelas a cada execução.

## 1. Imports e leitura

In [1]:
import json
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from db import get_engine

engine = get_engine()

silver = pd.read_sql("SELECT * FROM silver.custo_soja", engine)
print(f"Silver: {len(silver):,} linhas")

Silver: 23,727 linhas


## 2. `gold.fato_custo` — itens filtrados

Mantém apenas `tipo_linha = 'item'` e remove qualquer `Nao classificado`
remanescente (por segurança — não deveria haver item não classificado). A Gold é
enxuta: sem as colunas de rastreabilidade e a canônica, que eram da Silver.

In [2]:
fato = silver[silver["tipo_linha"] == "item"].copy()
fato = fato[fato["grupo_economico"] != "Nao classificado"]

COLS_FATO = [
    "local", "estado", "ano", "tipo_cultivo",
    "grupo_economico", "categoria", "descricao",
    "custo_ha", "custo_60kg", "produtividade_kg_ha",
]
fato = fato[COLS_FATO].reset_index(drop=True)

print(f"Fato: {len(fato):,} linhas (só itens classificados)")
print("Grupos:", sorted(fato["grupo_economico"].unique()))
assert (fato["grupo_economico"] == "Nao classificado").sum() == 0

Fato: 18,323 linhas (só itens classificados)
Grupos: ['Custos financeiros e seguros', 'Defensivos', 'Fertilizantes', 'Mao de obra', 'Mecanizacao', 'Pos-colheita', 'Sementes', 'Terra e outros']


## 3. `gold.dim_estado` — dimensão geográfica

Lê o GeoJSON dos estados e monta a dimensão. A geometria (polígono) fica como
texto GeoJSON na coluna `geometria` — o dashboard a consome direto (Plotly/Folium)
sem precisar de PostGIS.

In [3]:
GEOJSON = Path("..") / "dados" / "geo" / "br_estados.geojson"
assert GEOJSON.exists(), f"GeoJSON não encontrado: {GEOJSON.resolve()}"

geo = json.loads(GEOJSON.read_text(encoding="utf-8"))

dim_estado = pd.DataFrame([
    {
        "estado": f["properties"]["estado"],
        "nome_estado": f["properties"]["nome_estado"],
        "regiao": f["properties"]["regiao"],
        "codigo_ibge": f["properties"]["codigo_ibge"],
        "geometria": json.dumps(f["geometry"]),  # polígono como texto GeoJSON
    }
    for f in geo["features"]
])

print(f"dim_estado: {len(dim_estado)} estados")

# Integridade: todo estado do fato precisa ter geometria
orfaos = set(fato["estado"].unique()) - set(dim_estado["estado"].unique())
assert not orfaos, f"Estados sem geometria: {orfaos}"
print("Integridade OK — todo estado do fato tem geometria.")

dim_estado: 27 estados
Integridade OK — todo estado do fato tem geometria.


## 4. Gravar as tabelas Gold

In [4]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fato_custo;"))
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_estado;"))

fato.to_sql("fato_custo", schema="gold", con=engine,
            if_exists="replace", index=False, chunksize=5000, method="multi")
dim_estado.to_sql("dim_estado", schema="gold", con=engine,
                  if_exists="replace", index=False)

print(f"Gravado: gold.fato_custo ({len(fato):,}) | gold.dim_estado ({len(dim_estado)})")

Gravado: gold.fato_custo (18,323) | gold.dim_estado (27)


## 5. Validar — custo total por estado (pronto para o mapa)

Reconstrói o custo total por hectare (soma dos grupos por registro, média entre
registros do estado) e junta com a geometria. É exatamente a consulta que o
dashboard usará para o mapa coroplético.

In [5]:
query_mapa = text("""
    WITH custo_por_registro AS (
        SELECT estado, ano, local, tipo_cultivo,
               SUM(custo_ha) AS custo_total_ha
        FROM gold.fato_custo
        GROUP BY estado, ano, local, tipo_cultivo
    )
    SELECT e.estado, e.nome_estado, e.regiao,
           ROUND(AVG(c.custo_total_ha)::numeric, 2) AS custo_ha_medio
    FROM custo_por_registro c
    JOIN gold.dim_estado e ON e.estado = c.estado
    GROUP BY e.estado, e.nome_estado, e.regiao
    ORDER BY custo_ha_medio DESC
""")

pd.read_sql(query_mapa, engine)

,estado,nome_estado,regiao,custo_ha_medio
0,PI,Piauí,Nordeste,4381.80
1,RR,Roraima,Norte,4229.84
2,TO,Tocantins,Norte,4213.19
3,BA,Bahia,Nordeste,3644.04
4,DF,Distrito Federal,Centro-Oeste,3568.71
5,MA,Maranhão,Nordeste,3375.47
6,MG,Minas Gerais,Sudeste,3331.43
7,SP,São Paulo,Sudeste,3261.19
8,PR,Paraná,Sul,2963.20
9,MT,Mato Grosso,Centro-Oeste,2850.19


## 6. Validar — composição de custo por grupo

Participação média de cada grupo econômico no custo por hectare. Base para os
gráficos de composição do dashboard.

In [6]:
pd.read_sql(text("""
    SELECT grupo_economico,
           COUNT(*) AS linhas,
           ROUND(AVG(custo_ha)::numeric, 2) AS custo_ha_medio
    FROM gold.fato_custo
    GROUP BY grupo_economico
    ORDER BY custo_ha_medio DESC
"""), engine)

,grupo_economico,linhas,custo_ha_medio
0,Defensivos,515,521.22
1,Fertilizantes,1017,309.95
2,Sementes,1082,125.95
3,Terra e outros,2665,112.71
4,Mecanizacao,2486,63.93
5,Custos financeiros e seguros,5240,55.29
6,Pos-colheita,1972,31.81
7,Mao de obra,3346,26.21
